# Week 5 — Python สำหรับ Statistical Learning (Linear vs KNN, Test MSE)

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week05/`)
- `biz_customers.csv` — ลูกค้า 1203 คน
- `biz_campaigns.csv` — แคมเปญ 240 รายการ

## กติกาที่ต้องระวัง (จุดที่คนตกมักที่สุด)

1. **แบ่ง train/test ตามตำแหน่งแถว ไม่ใช่การสุ่ม**
   - ลูกค้า: train = 960 แถวแรก (`.iloc[:960]`), test = 243 แถวที่เหลือ (`.iloc[960:]`)
   - แคมเปญ: train = 192 แถวแรก (`.iloc[:192]`), test = 48 แถวที่เหลือ (`.iloc[192:]`)
   **ห้ามใช้ `train_test_split`** เพราะจะสุ่มแถวและคำตอบจะไม่ตรง
2. **ใช้ค่าเริ่มต้นของ scikit-learn ทุกอย่าง**
   (KNN ใช้ระยะทางยุคลิดและเฉลี่ยค่า y ของเพื่อนบ้านอย่างเท่ากัน)
3. **ไม่ต้อง standardize** ในสัปดาห์นี้ (ไม่มีข้อไหนสั่งให้ทำ)
4. **R² บนชุด test ใช้ค่าเฉลี่ยของ y ในชุด test** ไม่ใช่ของ train
   → ใช้ `model.score(X_test, y_test)` ซึ่งทำให้ถูกต้องให้เอง
5. **baseline ใช้ค่าเฉลี่ยของชุด train** (ไม่ใช่ชุด test) ค่าเดียว แล้วนำไปเทียบกับ y ของ test
6. **ลำดับตัวทำนายไม่มีผลต่อคำตอบ** แต่ให้เรียงตามโจทย์เพื่ออ่านง่าย
7. คำสั่งที่ใช้บ่อย: `LinearRegression()`, `KNeighborsRegressor(n_neighbors=K)`,
   `mean_squared_error(y_true, y_pred)`, `model.fit()`, `model.predict()`

> ทุก cell ข้อในไฟล์นี้**เป็นอิสระ** คัดลอกไปรันเดี่ยว ๆ ได้เลย
> (มี import และโหลดไฟล์ของตัวเองครบ) อ่านข้อไหนก็ดูข้อนั้นได้เลย

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week05')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ไฟล์ครบแล้ว: data\week05


In [2]:
# --- ตรวจไฟล์ข้อมูล (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd

DATA_DIR = Path(f'data/week05')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv']

for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์  {list(df.columns)}')

biz_campaigns.csv        240 แถว x  7 คอลัมน์  ['tv', 'radio', 'social', 'email', 'season', 'sales', 'total_budget']
biz_customers.csv       1203 แถว x 13 คอลัมน์  ['gender', 'age', 'income_k', 'tenure_months', 'channel', 'loyalty_member', 'web_visits', 'email_opens', 'basket_size', 'spend_k', 'orders_90d', 'churn', 'responded']


## ข้อ 5.1 — Test MSE ของโมเดลเชิงเส้น

- train = 960 แถวแรก, test = 243 แถวที่เหลือ (**แบ่งตามตำแหน่งแถว ไม่สุ่ม**)
- X = `income_k`, `tenure_months`, `email_opens`, y = `spend_k`
- Test MSE = ค่าเฉลี่ยของ (y จริง − y ทำนาย)² บนชุด test เท่านั้น

In [3]:
# --- ข้อ 5.1 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week05') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

cols = ['income_k', 'tenure_months', 'email_opens']
train, test = cust.iloc[:960], cust.iloc[960:]        # แบ่งตามตำแหน่งแถว ไม่สุ่ม

Xtr = train[cols].to_numpy(dtype=float)
Xte = test[cols].to_numpy(dtype=float)
ytr = train['spend_k'].to_numpy(dtype=float)
yte = test['spend_k'].to_numpy(dtype=float)

model = LinearRegression().fit(Xtr, ytr)             # fit ด้วยชุด train เท่านั้น
pred = model.predict(Xte)                            # ทำนายชุด test
answer = float(mean_squared_error(yte, pred))

print('train', Xtr.shape, '| test', Xte.shape)
print('coef =', np.round(model.coef_, 4), '| intercept =', round(float(model.intercept_), 4))
print('Test MSE =', answer)
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

train (960, 3) | test (243, 3)
coef = [0.0491 0.0075 0.0841] | intercept = 1.3068
Test MSE = 1.9486420092044723
ค่าที่กรอก: 3 ตำแหน่ง -> 1.949


## ข้อ 5.2 — ค่าเฉลี่ยแบบมีเงื่อนไข 3 ข้อพร้อมกัน

- `channel == 'online'` **และ** `loyalty_member == 0` **และ** `tenure_months > 54`
- ทั้งสามเงื่อนไขต้องเป็นจริง **พร้อมกัน** → ต่อด้วย `&` และใส่วงเล็บแต่เงื่อนไข
- `tenure_months > 54` = **มากกว่า** อย่างเคร่งครัด (ไม่ใช่ `>=`)

In [4]:
# --- ข้อ 5.2 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week05') / 'biz_customers.csv')   # ลูกค้า 1203 คน

mask = ((cust['channel'] == 'online')
        & (cust['loyalty_member'] == 0)
        & (cust['tenure_months'] > 54))             # ทั้งสามเงื่อนไขพร้อมกัน
answer = float(cust.loc[mask, 'basket_size'].mean())

print('จำนวนลูกค้าที่ผ่านทั้ง 3 เงื่อนไข =', int(mask.sum()))
print('ถ้าใช้ >= 54 จะได้จำนวน', int(((cust['channel'] == 'online') & (cust['loyalty_member'] == 0) & (cust['tenure_months'] >= 54)).sum()))
print('ค่าเฉลี่ย basket_size =', answer)
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

จำนวนลูกค้าที่ผ่านทั้ง 3 เงื่อนไข = 40
ถ้าใช้ >= 54 จะได้จำนวน 41
ค่าเฉลี่ย basket_size = 3.07
ค่าที่กรอก: 3 ตำแหน่ง -> 3.070


## ข้อ 5.3 — ค่าเฉลี่ยของค่าทำนายจากสูตรที่โจทย์กำหนด

- X = `camp.iloc[34:39]` (5 แถว) คอลัมน์ `tv`, `radio`, `social`
- ŷ = 3.6 + 0.04·tv + 0.14·radio + 0.02·social
- **ไม่ต้อง fit โมเดล** แค่คำนวณตามสูตร แล้วเฉลี่ยทั้ง 5 แถว
- `X @ β + b₀` คือการคูณจุดแล้วบวกค่าคงที่ (β เรียงตามลำดับคอลัมน์ของ X)

In [5]:
# --- ข้อ 5.3 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week05') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

X = camp.iloc[34:39][['tv', 'radio', 'social']].to_numpy(dtype=float)   # 5x3
beta = np.array([0.04, 0.14, 0.02])
yhat = X @ beta + 3.6
answer = float(yhat.mean())

print('X =')
print(np.array2string(X, precision=1, suppress_small=True))
print('yhat =', np.round(yhat, 3))
print('ค่าเฉลี่ย =', answer)
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

X =
[[210.4  24.9  22.2]
 [139.3  25.8  46.7]
 [162.2  35.8  29.6]
 [ 64.6  24.9  28. ]
 [ 88.6   9.1  13.3]]
yhat = [15.946 13.718 15.692 10.23   8.684]
ค่าเฉลี่ย = 12.854
ค่าที่กรอก: 3 ตำแหน่ง -> 12.854


## ข้อ 5.4 — สัมประสิทธิ์สหสัมพันธ์ Pearson เฉพาะไตรมาส Q1

- กรองเฉพาะแคมเปญที่ `season == 'Q1'` **ก่อน** แล้วคำนวณ correlation ของ `tv` กับ `radio`
- ค่าเดียวกับ `df.corr()` ของ pandas (Pearson เป็นค่าเริ่มต้น)

In [6]:
# --- ข้อ 5.4 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week05') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

q1 = camp[camp['season'] == 'Q1']            # กรองก่อน แล้วค่อยหา correlation
answer = float(q1['tv'].corr(q1['radio']))

print('จำนวนแคมเปญ Q1 =', len(q1))
print('(ถ้าไม่กรอง: corr ทั้งไฟล์ =', round(float(camp['tv'].corr(camp['radio'])), 4), '-> ผิด)')
print('corr(tv, radio) ใน Q1 =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนแคมเปญ Q1 = 67
(ถ้าไม่กรอง: corr ทั้งไฟล์ = -0.0301 -> ผิด)
corr(tv, radio) ใน Q1 = 0.1877337687257459
ค่าที่กรอก: 4 ตำแหน่ง -> 0.1877


## ข้อ 5.5 — จำนวนเพื่อนบ้าน K ที่ทำให้ Test MSE ต่ำที่สุด

- train = 192 แถวแรก, test = 48 แถวที่เหลือ
- X = `tv`, `social` (ไม่ standardize), y = `sales`
- ลอง K = 1..60 ทีละค่า แล้วหาค่าที่ Test MSE ต่ำสุด

In [7]:
# --- ข้อ 5.5 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week05') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

cols = ['tv', 'social']
train, test = camp.iloc[:192], camp.iloc[192:]

Xtr = train[cols].to_numpy(dtype=float)
Xte = test[cols].to_numpy(dtype=float)
ytr = train['sales'].to_numpy(dtype=float)
yte = test['sales'].to_numpy(dtype=float)

rows = []
for K in range(1, 61):
    knn = KNeighborsRegressor(n_neighbors=K).fit(Xtr, ytr)
    rows.append((K, mean_squared_error(yte, knn.predict(Xte))))

best_K, best_mse = min(rows, key=lambda r: r[1])      # ค่า K ที่ Test MSE ต่ำสุด

print('train', Xtr.shape, '| test', Xte.shape)
print('K ที่ดีที่สุด =', best_K, '| Test MSE =', round(best_mse, 6))
print('เทียบ K ใกล้เคียง:', [(k, round(m, 4)) for k, m in rows[best_K - 3:best_K + 2]])
print('ค่าที่กรอก: จำนวนเต็ม ->', best_K)

train (192, 2) | test (48, 2)
K ที่ดีที่สุด = 20 | Test MSE = 29.638591
เทียบ K ใกล้เคียง: [(18, 29.6576), (19, 29.9485), (20, 29.6386), (21, 30.1234), (22, 30.3269)]
ค่าที่กรอก: จำนวนเต็ม -> 20


## ข้อ 5.6 — R² บนชุด test

- train = 960 แถวแรก, test = 243 แถวที่เหลือ
- X = `loyalty_member`, `basket_size`, y = `orders_90d`
- **ค่าเฉลี่ยที่ใช้ใน R² คือค่าเฉลี่ยของ y ในชุด test**
  → ใช้ `model.score(X_test, y_test)` จะได้ถูกต้องให้เอง

In [8]:
# --- ข้อ 5.6 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week05') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.linear_model import LinearRegression

cols = ['loyalty_member', 'basket_size']
train, test = cust.iloc[:960], cust.iloc[960:]

Xtr = train[cols].to_numpy(dtype=float)
Xte = test[cols].to_numpy(dtype=float)
ytr = train['orders_90d'].to_numpy(dtype=float)
yte = test['orders_90d'].to_numpy(dtype=float)

model = LinearRegression().fit(Xtr, ytr)
answer = float(model.score(Xte, yte))          # R^2 บนชุด test

print('coef =', np.round(model.coef_, 4), '| intercept =', round(float(model.intercept_), 4))
print('ค่าเฉลี่ย y ในชุด test =', round(float(yte.mean()), 4), '(นี่คือค่าเฉลี่ยที่ R2 ใช้)')
print('R2 (test) =', answer)
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

coef = [0.9361 0.0269] | intercept = 1.2863
ค่าเฉลี่ย y ในชุด test = 1.6008 (นี่คือค่าเฉลี่ยที่ R2 ใช้)
R2 (test) = 0.118126684994039
ค่าที่กรอก: 4 ตำแหน่ง -> 0.1181


## ข้อ 5.7 — ทำนายยอดขายของแคมเปญใหม่

- fit `LinearRegression` ด้วย **192 แถวแรก** ของแคมเปญ, X = `tv`, `radio`, y = `sales`
- แคมเปญใหม่: tv = 204.4, radio = 41.7
- ทำนายด้วย `model.predict([[204.4, 41.7]])` → ต้องใส่เป็นรูป 2 มิติ (1 แถว 2 คอลัมน์)

In [9]:
# --- ข้อ 5.7 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week05') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ
from sklearn.linear_model import LinearRegression

train = camp.iloc[:192]
Xtr = train[['tv', 'radio']].to_numpy(dtype=float)
ytr = train['sales'].to_numpy(dtype=float)

model = LinearRegression().fit(Xtr, ytr)
new_campaign = [[204.4, 41.7]]                 # 1 แถว 2 คอลัมน์
answer = float(model.predict(new_campaign)[0])

print('แคมเปญใหม่: tv = 204.4, radio = 41.7')
print('coef =', np.round(model.coef_, 4), '| intercept =', round(float(model.intercept_), 4))
print('ยอดขายที่ทำนาย =', answer, 'พันชิ้น')
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

แคมเปญใหม่: tv = 204.4, radio = 41.7
coef = [0.0746 0.3529] | intercept = -1.504
ยอดขายที่ทำนาย = 28.462743325247974 พันชิ้น
ค่าที่กรอก: 3 ตำแหน่ง -> 28.463


## ข้อ 5.8 — โมเดลเชิงเส้นคุ้มกว่าโมเดลพื้นฐานไหม

- baseline ทำนาย `spend_k` ทุกคนด้วย **ค่าเฉลี่ยของ `spend_k` ในชุด train** (ค่าเดียว)
- โมเดลเชิงเส้น: X = `income_k`, `web_visits`, `basket_size`, y = `spend_k`
- คำตอบ = Test MSE ของเส้น ÷ Test MSE ของ baseline
- ค่านี้ **น้อยกว่า 1** แปลว่าโมเดลเชิงเส้นดีกว่า

In [10]:
# --- ข้อ 5.8 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week05') / 'biz_customers.csv')   # ลูกค้า 1203 คน
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

cols = ['income_k', 'web_visits', 'basket_size']
train, test = cust.iloc[:960], cust.iloc[960:]

Xtr = train[cols].to_numpy(dtype=float)
Xte = test[cols].to_numpy(dtype=float)
ytr = train['spend_k'].to_numpy(dtype=float)
yte = test['spend_k'].to_numpy(dtype=float)

model = LinearRegression().fit(Xtr, ytr)
mse_linear = mean_squared_error(yte, model.predict(Xte))

baseline = ytr.mean()                          # ค่าเฉลี่ยของ TRAIN ไม่ใช่ test
mse_base = mean_squared_error(yte, np.full_like(yte, baseline))
answer = float(mse_linear / mse_base)

print('baseline (ค่าเฉลี่ยของ train) =', round(float(baseline), 4))
print('Test MSE เส้น  =', round(mse_linear, 6))
print('Test MSE base  =', round(mse_base, 6))
print('อัตราส่วน =', answer, '(น้อยกว่า 1 = เส้นดีกว่า)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

baseline (ค่าเฉลี่ยของ train) = 3.4656
Test MSE เส้น  = 1.761803
Test MSE base  = 2.353777
อัตราส่วน = 0.7485002064030836 (น้อยกว่า 1 = เส้นดีกว่า)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.7485


## ข้อ 5.9 — KNN(k=9) ทำนายแคมเปญแถวที่ 211

- fit ด้วย **192 แถวแรก**, X = `tv`, `radio`, y = `sales`
- แคมเปญแถวที่ 211 อยู่ในชุด test → ใช้ค่าของแถวนั้นเป็นจุดที่จะถาม
- KNN ทำนายด้วยค่าเฉลี่ย `sales` ของ 9 แคมเปญใกล้เคียงที่สุด

In [11]:
# --- ข้อ 5.9 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week05') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ
from sklearn.neighbors import KNeighborsRegressor

train = camp.iloc[:192]
Xtr = train[['tv', 'radio']].to_numpy(dtype=float)
ytr = train['sales'].to_numpy(dtype=float)

knn = KNeighborsRegressor(n_neighbors=9).fit(Xtr, ytr)

target = camp.iloc[211]                        # อยู่ในชุด test
answer = float(knn.predict([[target['tv'], target['radio']]])[0])

print('แถว 211: tv =', target['tv'], ', radio =', target['radio'], ', sales จริง =', target['sales'])
print('ยอดขายที่ KNN(k=9) ทำนาย =', answer)
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

แถว 211: tv = 106.8 , radio = 25.7 , sales จริง = 14.12
ยอดขายที่ KNN(k=9) ทำนาย = 15.44222222222222
ค่าที่กรอก: 3 ตำแหน่ง -> 15.442


## ข้อ 5.10 — นับแคมเปญที่ KNN ทำนายแม่นกว่าเส้น

- train = 192 แถวแรก, test = 48 แถวที่เหลือ
- ทั้งสองโมเดลใช้ X = `tv`, `radio`, `email` และ y = `sales`
- เทียบกัน **รายแคมเปญ** ในชุด test: นับที่ `|KNN − จริง| < |เส้น − จริง|`
  เงื่อนไข **น้อยกว่าอย่างเคร่งครัด** (เท่ากันไม่นับ)

In [12]:
# --- ข้อ 5.10 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week05') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor

cols = ['tv', 'radio', 'email']
train, test = camp.iloc[:192], camp.iloc[192:]

Xtr = train[cols].to_numpy(dtype=float)
Xte = test[cols].to_numpy(dtype=float)
ytr = train['sales'].to_numpy(dtype=float)
yte = test['sales'].to_numpy(dtype=float)

lin = LinearRegression().fit(Xtr, ytr)
knn = KNeighborsRegressor(n_neighbors=7).fit(Xtr, ytr)

err_lin = np.abs(yte - lin.predict(Xte))
err_knn = np.abs(yte - knn.predict(Xte))
answer = int((err_knn < err_lin).sum())

print('จำนวนแคมเปญในชุด test =', len(yte))
print('KNN แม่นกว่า =', answer, '| เท่ากัน =', int((err_knn == err_lin).sum()),
      '| KNN แย่กว่า =', int((err_knn > err_lin).sum()))
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

จำนวนแคมเปญในชุด test = 48
KNN แม่นกว่า = 17 | เท่ากัน = 0 | KNN แย่กว่า = 31
ค่าที่กรอก: จำนวนเต็ม -> 17


## สรุปคำตอบทั้ง 10 ข้อ (Week 5)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อน` (ไม่มีการผ่อนปรนเพิ่ม)

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 5.1 | Test MSE ของ LinearRegression (960/243) | **1.949** | 3 ตำแหน่ง | 0.0015 |
| 5.2 | ค่าเฉลี่ย basket_size (3 เงื่อนไขพร้อมกัน) | **3.070** | 3 ตำแหน่ง | 0.0015 |
| 5.3 | ค่าเฉลี่ยของ yhat จากสูตรที่กำหนด | **12.854** | 3 ตำแหน่ง | 0.0015 |
| 5.4 | corr(tv, radio) เฉพาะไตรมาส Q1 | **0.1877** | 4 ตำแหน่ง | 0.00015 |
| 5.5 | K ที่ทำให้ Test MSE ต่ำที่สุด (k=1..60) | **20** | จำนวนเต็ม | 0.5 |
| 5.6 | R2 บนชุด test (960/243) | **0.1181** | 4 ตำแหน่ง | 0.00015 |
| 5.7 | ยอดขายแคมเปญใหม่ tv=204.4 radio=41.7 | **28.463** | 3 ตำแหน่ง | 0.0015 |
| 5.8 | Test MSE เส้น / Test MSE baseline | **0.7485** | 4 ตำแหน่ง | 0.00015 |
| 5.9 | KNN(k=9) ทำนายแคมเปญแถวที่ 211 | **15.442** | 3 ตำแหน่ง | 0.0015 |
| 5.10 | จำนวนแคมเปญที่ KNN แม่นกว่าเส้น | **17** | จำนวนเต็ม | 0.5 |

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน